In [ ]:
############################################## SETTINGS ##############################################  
sim = 'sim2256'

# settings
cell_type='ispn'
model = 3

import os, sys
# compute absolute path of main folder
cwd = os.getcwd()

if os.path.exists(os.path.join(cwd, 'settings.py')):
    main_dir = cwd
else:
    main_dir = os.path.abspath(os.path.join(cwd, '..'))

# set CWD or add to path
os.chdir(main_dir)
sys.path.insert(0, main_dir)

# then import/run settings
%run -i settings.py

from analysis_functions import *

# Get the current working directory
current_wd = os.getcwd()
# 'sim' + str(sim)

downsample = True

home = os.path.expanduser('~')

# Set working directory
external = False
external_name = 'MacOS10'
target = f'model{model}'

if not external:
    if downsample:
        wd = home + '/Documents/Repositories/msNEURON_Belal2026/downsample/' + cell_type + '/' + target + '/physiological/simulations'
    else:
        wd = home + '/Documents/Repositories/msNEURON_Belal2026/' + cell_type + '/' + target + '/physiological/simulations'
else:
    if downsample:
        wd = '/Volumes/' + external_name + '/Repositories/msNEURON_Belal2026/downsample/' + cell_type + '/' + target + '/physiological/simulations'
    else:
        wd = '/Volumes/' + external_name + '/Repositories/msNEURON_Belal2026/' + cell_type + '/' + target + '/physiological/simulations'


# create path to directory to save analysis
base_path = os.path.join(home, 'Documents', 'Repositories', 'msNEURON_Belal2026', 'analysis')
sim_image_path = os.path.join(base_path, cell_type, sim)
os.makedirs(sim_image_path, exist_ok=True)

save = True

<h3 style="text-align: center;">Basic Graph Settings</h3>

**s = 5** 

splprep fits a parametric B-spline to the 2D points 
s= 5 provides moderate smoothing:

- doesn't distort genuine bends or branch shapes,
- suppresses small coordinate noise or jaggedness that often appears in morphology data.

**lwd = 1**

For **morphology** and **heatmaps**:

**height = 600**

**width = 600**

For **voltage** and **current density** traces:

**height = 600**

**width = 800**

In [ ]:
s = 5 # splprep fits a parametric B-spline to the 2D points 
lwd = 1 # standardise line width

height1 = 600
width1 = 600

height2 = 600
width2 = 800

if cell_type == 'ispn':
    plane='yx'
    mirror=False
    dend_name = 'dend[12]'
    
else:
    plane='xy'
    mirror=False
    dend_name = 'dend[7]'

if downsample:
    ds = 10
    


<h3 style="text-align: center;">Data Loading Function</h3>

<span style="font-family: monospace; font-weight: 550;">
load_data_dicts(parent_dir)
</span><br>
Loads simulation data from a structured directory where each subfolder represents a simulation (e.g., <i>a</i>, <i>b</i>, <i>c</i>) containing multiple pickle files.<br>
Inputs: <i>parent_dir</i> (path to directory containing simulation folders)<br>
Returns: <i>Dictionary of simulation data, structured as {simulation: {variable_name: data}}</i>

<hr style="margin: 6px 0;">

<span style="font-family: monospace; font-weight: 550;">
sims_dir = os.path.join(wd, sim)<br>
sim_data = load_data_dicts(sims_dir)
</span><br>
Combines data from all simulation subdirectories into a single nested dictionary for convenient access.

<br>

<i>Example structure:</i><br>
sim_data → sim_letter (e.g. 'a', 'b', 'c', …) → variable_name (e.g. 'v_all_3D', 'metadata', 'i_nmda', …) → content (arrays, dicts, DataFrames, etc.)

<h3 style="text-align: center;">Data Loading with Automatic Caching</h3>

Loads simulation data from nested directories of <span style="font-family: monospace; font-weight: 550;">.pickle</span> files, with optional automatic caching for external drives.<br>
When the data resides on an external volume (e.g., USB or external SSD), the function copies it once to a local cache in <i>~/Documents/simcache</i> for faster access in future sessions.
<br>

<span style="font-family: monospace; font-weight: 550;">load_data_dicts(wd, sim, cell_type=None, copy_to_cache=True, cache_dir=None, verbose=False)</span><br>
If <i>copy_to_cache</i> is enabled and <i>os.path.join(wd, sim)</i> is detected on an external drive, the data are transferred to the local cache and then loaded.<br>
Subsequent calls automatically detect the cached copy and load from there instead of repeating the transfer.<br>
All subfolders (e.g. <i>a</i>, <i>b</i>, <i>c</i>) are read and combined into a nested dictionary keyed by simulation name.<br>
If no subfolders are found, all <span style="font-family: monospace; font-weight: 550;">.pickle</span> files in the simulation directory are loaded directly.
<br>

*Inputs*<br>
<i>wd</i>: path to the parent simulations directory<br>
<i>sim</i>: simulation identifier, e.g. <span style="font-family: monospace; font-weight: 550;">sim2256</span><br>
<i>cell_type</i>: optional cell type used to organize the cache directory<br>
<i>copy_to_cache</i>: if True, automatically cache external data in <span style="font-family: monospace; font-weight: 550;">~/Documents/simcache/&lt;cell_type&gt;</span> (default True)<br>
<i>cache_dir</i>: optional path to custom cache location<br>
<i>verbose</i>: print progress and timing information during copy and load operations
<br>

*Returns*<br>
<i>Nested dictionary</i> structured as <span style="font-family: monospace; font-weight: 550;">{simulation: {variable_name: data}}</span> or, if no subfolders are found, a flat dictionary of <span style="font-family: monospace; font-weight: 550;">{variable_name: data}</span>
<br>

Example usage:<br>
<span style="font-family: monospace; font-weight: 550;">sim_data = load_data_dicts(wd=wd, sim=sim, cell_type=cell_type, verbose=True)</span>
<br>

Example console output:<br>
<span style="font-family: monospace; font-weight: 550;">
External drive detected: copying to cache...<br>
From: /Volumes/MacOS10/Repositories/msNEURON_Belal2026/dspn/model3/physiological/simulations/sim2411<br>
To:   ~/Documents/simcache/dspn/sim2411<br>
Copying:  6982.0 / 11200.7 MB ( 62.3%)  Elapsed:  12.7s  Remaining:   7.3s<br>
Copy complete in 20.1 s<br><br>
All data loaded from ~/Documents/simcache/dspn/sim2411
</span>
<br>

Once the cache is created, subsequent runs automatically detect and use it:<br>
<span style="font-family: monospace; font-weight: 550;">
Using cached copy at ~/Documents/simcache/dspn/sim2411<br>
All data loaded from ~/Documents/simcache/dspn/sim2411
</span>

In [ ]:
# load simulations
sims_dir = os.path.join(wd, sim)

sim_data = load_data_dicts(wd=wd, sim=sim, cell_type=cell_type, verbose=True)

In [ ]:
# check files loaded in correct order
for name in sim_data.keys():
    print(name)

<h3 style="text-align: center;">Memory Usage Reporting</h3>

<span style="font-family: monospace; font-weight: 550;">
report_memory(verbose=False)
</span><br>
Reports current notebook kernel and overall system memory usage.<br>
Inputs: <i>verbose</i>: optional; if True, prints PID and memory percentages<br>
Returns: <i>None</i>; prints memory summary to console

<br>

Example:
```python
report_memory()
report_memory(verbose=True)

In [ ]:
report_memory(verbose=True)

<h3 style="text-align: center;">Simulation Data Structure</h3>

<span style="font-family: monospace; font-weight: 550;">
sim_data
</span><br>
Nested dictionary containing simulation outputs, where each subfolder (e.g., <i>a</i>, <i>b</i>, <i>c</i>) represents a separate simulation run.<br><br>

Structure:
<pre>
sim_data
  └ sim_letter (e.g. 'a', 'b', 'c', ...)
      └ variable_name (e.g. 'v_all_3D', 'metadata', 'i_nmda', ...)
          └ content (lists, dicts, arrays, DataFrames, etc.)
</pre>

To view the available variables from the first simulation:
```python
list(next(iter(sim_data.values()), {}).keys())
```

To obtain the cell coordinates from the first simulation:
```python
next(iter(sim_data.values()))['mechanisms_3D_distribution']['cell_coordinates']    
```

<h3 style="text-align: center;">Morphology and Coordinate Functions</h3>

<span style="font-family: monospace; font-weight: 550">
morphology_plot(cell_coordinates, dend_tree, lwd=0.8, color='black', s=None, height=600, width=600, scale_bar=50, x_range=[-125, 175], y_range=[-150, 150], plane='xy', mirror=False, idxs=None, idxs_colors=None, alpha=0.5, dot_size=4)
</span><br>
Plots neuronal morphology and optionally overlays synapse locations or other indexed points.<br>
Inputs: <i>cell_coordinates, dend_tree, optional idxs, idxs_colors, dot_size, alpha, plane, mirror, scale_bar, x_range, y_range, lwd, color, s</i><br>
Returns: <i>Interactive Plotly figure showing full neuron morphology</i>

<hr style="margin: 6px 0;">

<span style="font-family: monospace; font-weight: 550">
morphology_plot1(cell_coordinates, dend_tree, lwd=0.8, color='black', height=600, width=600, scale_bar=50, x_range=[-125, 175], y_range=[-150, 150], plane='xy', mirror=False)
</span><br>
<span style="font-family: monospace; font-weight: 550">
morphology_plot2(cell_coordinates, dend_tree, lwd=0.8, color='black', s=2, height=600, width=600, scale_bar=50, x_range=[-125, 175], y_range=[-150, 150], plane='xy', mirror=False)
</span><br>
Internal helpers for morphology_plot(): morphology_plot1 draws straight connections; morphology_plot2 uses smoothed spline curves for dendritic paths.<br>
Inputs: <i>Morphology data, dendritic tree, rendering options</i><br>
Returns: <i>Plotly figure object</i>

<hr style="margin: 6px 0;">

<span style="font-family: monospace; font-weight: 550">
get_plane_coords(coords, plane='xy', mirror=False)
</span><br>
Projects 3D morphology coordinates onto a 2D plane for visualisation.<br>
Inputs: <i>coords (array), plane ('xy', 'xz', or 'yz'), mirror (bool)</i><br>
Returns: <i>2D NumPy array of [x, y] coordinates</i>

<hr style="margin: 6px 0;">

<span style="font-family: monospace; font-weight: 550">
get_coord_index(cell_coordinates, target_dendrite, target_location)
</span><br>
Finds nearest indices in cell_coordinates for given dendrite(s) and relative position(s).<br>
Inputs: <i>cell_coordinates, target_dendrite, target_location</i><br>
Returns: <i>Integer index or list of indices</i>

<hr style="margin: 6px 0;">

<span style="font-family: monospace; font-weight: 550">
get_coord_index_interp(cell_coordinates, target_dendrite, target_location)
</span><br>
Interpolates 3D coordinates along dendrites, enabling smooth synapse placement.<br>
Inputs: <i>cell_coordinates, target_dendrite, target_location</i><br>
Returns: <i>Array [section, rel_x, x, y, z]</i>

<hr style="margin: 6px 0;">

Example:
```python
coords = get_coord_index_interp(cell_coordinates, ['dend[15]', 'dend[42]'], [0.7, 0.3])
indices = get_coord_index(cell_coordinates, ['dend[15]', 'dend[42]'], [0.7, 0.3])
fig = morphology_plot(cell_coordinates, dend_tree, idxs=[coords], idxs_colors=['#CD5C5C'], dot_size=5)
fig.show()

<h3 style="text-align: center;">Cell Coordinate Summary</h3>

<span style="font-family: monospace; font-weight: 550">
summarise_cell_coordinates(sim_data)
</span><br>
Scans simulation data for <i>cell_coordinates</i> arrays, reports which variables contain them, checks cross-simulation consistency, and returns a boolean indicating whether 3D morphology data exist.<br>
Inputs: <i>sim_data (nested dictionary of simulation outputs)</i><br>
Returns: <i>Boolean — True if 3D coordinates exist, False otherwise</i>

<hr style="margin: 6px 0;">

Example:
```python
coords_exist = summarise_cell_coordinates(sim_data)

if coords_exist:
    first_sim = next(iter(sim_data.values()))
    v_all_3D = first_sim['v_all_3D']
    first_key = next(iter(v_all_3D))
    cell_coordinates = np.array(v_all_3D[first_key]['cell_coordinates'], dtype=object)
    _, _, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True,
                                         spine_per_length=spine_per_length, frequency=frequency,
                                         d_lambda=d_lambda, verbose=False, dend2remove=None)
else:
    cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True,
                                         spine_per_length=spine_per_length, frequency=frequency,
                                         d_lambda=d_lambda, verbose=False, dend2remove=None)
    cell_coordinates = []
    for sec in cell.somalist:
        h('access ' + sec.name())
        x, y, z, diam = interpolate_3d(sec, 0.5)
        cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])
    for sec in cell.dendlist:
        for seg in sec:
            x, y, z, diam = interpolate_3d(sec, seg.x)
            cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])
    cell_coordinates = np.array(cell_coordinates, dtype=object)

In [ ]:
# check if coordinates exist
coords_exist = summarise_cell_coordinates(sim_data)

# if so load from v_all_3D else retrieve from cell_build
# this is because the methods to retrieve the cell_coordinates differ so it is important when plotting 3D heatmaps that the correct coordinates are used
# the method to default will only be used when the simulations have no 3d data 
if coords_exist:
    first_sim = next(iter(sim_data.values()))
    v_all_3D = first_sim['v_all_3D']
    first_key = next(iter(v_all_3D)) 
    
    cell_coordinates = np.array(v_all_3D[first_key]['cell_coordinates'], dtype=object)
    _, _, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=None)
else:
    cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=None)
    cell_coordinates = []
    
    for sec in cell.somalist:
        h('access ' + sec.name())
        x, y, z, diam = interpolate_3d(sec, 0.5)  # Use 0.5 to refer to the center of the section
        cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])
    
    # Setup for dendritic sections
    for sec in cell.dendlist:
        for seg in sec:
            x, y, z, diam = interpolate_3d(sec, seg.x)
            cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])
        
    cell_coordinates = np.array(cell_coordinates, dtype=object)

    
fig_morphology = morphology_plot(cell_coordinates, dend_tree, lwd=lwd, s=s, color='grey', height=height1, width=width1, plane=plane, mirror=False)

fig_morphology.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    fig_morphology.write_image(f"{sim_image_path}/morphology.svg", format='svg', scale=3)


<h3 style="text-align: center;">Adding Synapse Locations</h3>

<i>Helper functions to locate or interpolate given locations (e.g.synapses) along dendritic segments within a reconstructed neuron morphology.</i><br>

<span style="font-family: monospace; font-weight: 550; color: black;">
get_coord_index(cell_coordinates, target_dendrite, target_location)
</span><br>
Finds the nearest index (or indices) in <i>cell_coordinates</i> for specified dendritic locations.<br>
Inputs: <i>cell_coordinates</i> (NumPy array containing morphology data [section, x, y, z, ...]), <i>target_dendrite</i> (e.g. 'dend[15]' or list of names), <i>target_location</i> (relative position(s) 0–1)<br>
Returns: Integer index or list of indices corresponding to the nearest row(s) in <i>cell_coordinates</i>.

<hr style="margin: 8px 0;">

<span style="font-family: monospace; font-weight: 550; color: black;">
get_coord_index_interp(cell_coordinates, target_dendrite, target_location)
</span><br>
Interpolates 3D coordinates (<i>x</i>, <i>y</i>, <i>z</i>) for arbitrary locations along dendritic sections, providing smooth spatial mapping between recorded points.<br>
Inputs: <i>cell_coordinates</i> (morphology coordinate array), <i>target_dendrite</i> (section name(s)), <i>target_location</i> (fractional distance(s) along each section)<br>
Returns: Array of interpolated coordinates in the format [section, rel_x, x, y, z].

<hr style="margin: 8px 0;">

Example:
```python
coords = get_coord_index_interp(cell_coordinates, ['dend[15]', 'dend[42]'], [0.7, 0.3])
indices = get_coord_index(cell_coordinates, ['dend[15]', 'dend[42]'], [0.7, 0.3])

#### Neuronal Morphology


Flatten in Illustrator
- Select all (Cmd+A) → Object → Expand Appearance → then Ungroup repeatedly
- Delete the transparent rectangles

In [ ]:
# Morphology plot with synaptic locations

# extract names and locations of GLUT and GABA inputs from metadata
# using last simulation to get all GABA locations used (fixed)
dend_gaba_all = list(sim_data.values())[-1]['metadata']['dend_gaba']
gaba_locations_all = list(sim_data.values())[-1]['metadata']['gaba_locations']

dend_glut = list(sim_data.values())[-1]['metadata']['dend_glut']
glut_locations = list(sim_data.values())[-1]['metadata']['glutamate_locations']

if len(dend_glut) != len(glut_locations): # one upstate site 
    dend_glut = dend_glut * len(glut_locations)
    
# indices
idxs_gaba = get_coord_index(cell_coordinates=cell_coordinates, target_dendrite=dend_gaba_all, target_location=gaba_locations_all)
idxs_glut = get_coord_index(cell_coordinates=cell_coordinates, target_dendrite=dend_glut, target_location=glut_locations)

# coordinates
coords_gaba = get_coord_index_interp(cell_coordinates=cell_coordinates, target_dendrite=dend_gaba_all, target_location=gaba_locations_all)
coords_glut = get_coord_index_interp(cell_coordinates=cell_coordinates, target_dendrite=dend_glut, target_location=glut_locations)

# with interpolation
fig_morphology = morphology_plot(cell_coordinates=cell_coordinates, dend_tree=dend_tree, idxs=[coords_gaba, coords_glut], 
                                 title='', idxs_colors = ['#5393CF', '#CD5C5C'], dot_size=[6,6], lwd=lwd, s=s, color='grey', 
                                 height=height1, width=width1, plane=plane, mirror=mirror, alpha=[0.5, 0.25])

fig_morphology.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    fig_morphology.write_image(f"{sim_image_path}/morphology2.svg", format='svg', scale=3)

<h3 style="text-align: center;">Comparison of Simulation and Interpolated Coordinates</h3>

Compares the <i>simulation set values</i> (e.g. <span style="font-family: monospace; font-weight: 550">dend_gaba_all</span>, <span style="font-family: monospace; font-weight: 550;">gaba_locations_all</span>, <span style="font-family: monospace; font-weight: 550;">dend_glut</span>, <span style="font-family: monospace; font-weight: 550;">glut_locations</span>) with those obtained from the interpolated function <span style="font-family: monospace; font-weight: 550;">get_coord_index_interp</span>.<br>
The interpolated function estimates the 3D coordinates (<i>x</i>, <i>y</i>, <i>z</i>) along each dendritic section based on the relative position (<i>loc</i>) within that dendrite.

<br>

<span style="font-family: monospace; font-weight: 550;">compare_synapse_coords(title, dend_locs, rel_locs, coords)</span><br>
Displays side-by-side comparisons of simulation-based and interpolated coordinates for synaptic inputs, using formatted HTML for clarity.<br>
Inputs: <i>title</i> (string label for synapse type), <i>dend_locs</i> (list of dendrite names), <i>rel_locs</i> (list of relative positions 0–1), <i>coords</i> (list of tuples containing section, loc, x, y, z).<br>
Returns: None (outputs formatted HTML display within the notebook).

<br>

Example:
```python
compare_synapse_coords("Glutamatergic", dend_glut, glut_locations, coords_glut)
compare_synapse_coords("GABAergic", dend_gaba_all, gaba_locations_all, coords_gaba)



In [ ]:
compare_synapse_coords("Glutamatergic", dend_glut, glut_locations, coords_glut)

In [ ]:
compare_synapse_coords("GABAergic", dend_gaba_all, gaba_locations_all, coords_gaba)

<h3 style="text-align: center;">plot3_dt</h3>

Helper function to plot any variable in time (dt) (e.g. voltage, impedance or current densities) from simulation output arrays.  

<br>

<span style="font-family: monospace; font-weight: 550;">plot3_dt(Varray, title=None, yaxis='V (mV)', yrange=[-86, -60], stim_time=150, sim_time=400, black_trace=None, gray_trace=None, x_err_bar=100, y_err_bar=10, y_err_bar_shift=5, palette='oleron', alpha=0.8, reverse=False, baseline=20, dt=0.025, width=1000, height=400, black_shift=200, ds=10, offset=False, offset_ms=None, offset_y=None, lwd=1)</span>  

<br>

*Inputs*  
- <i>Varray</i>: list of voltage traces (each entry can be an array or a list containing an array)  
- <i>title</i>: optional string for figure title  
- <i>yaxis</i>: axis label, e.g. ‘V (mV)’ or ‘Impedance (MΩ)’  
- <i>yrange</i>: y-axis limits for all traces  
- <i>stim_time</i>: stimulation time (ms)  
- <i>sim_time</i>: total simulation duration (ms)  
- <i>black_trace</i>: index of trace to display in black (optional)  
- <i>gray_trace</i>: index of trace to display in gray (optional)  
- <i>x_err_bar</i>, <i>y_err_bar</i>: corner error bar scale indicators  
- <i>baseline</i>: baseline period before stimulation (ms)  
- <i>dt</i>: simulation timestep (ms)  
- <i>ds</i>: down-sampling factor for plotting  
- <i>offset</i>: logical; apply offset between traces if TRUE  
- <i>offset_ms</i>: horizontal shift between traces when offset is TRUE  
- <i>offset_y</i>: vertical shift between traces when offset is TRUE  
- <i>width</i>, <i>height</i>: figure dimensions  
- <i>lwd</i>: line width  

<br>

*Returns*  
- <i>fig_master</i>: Plotly figure.  

<br>

*Example*  
```python
plot3_dt(Vsoma, yaxis='V (mV)', stim_time = 150, sim_time=sim_time, title='somatic voltage', 
                x_err_bar=50, baseline=50, dt=dt, ds=10, offset=True, offset_ms=150)        


In [ ]:
# Simple output traces
# downsample plots if original simulation is sampled at high rate eg 0.025 ms
# if simulation was subsequently downsampled then no need to downsample plots

dt = next(iter(sim_data.values()))['metadata']['dt']

if downsample:
    ds_plot = 1
    dt = ds * dt
    
else:
    ds_plot = 10


Vsoma = []
Vdend = []

for variables in sim_data.values():
    if 'vsoma' in variables:
        Vsoma.append(variables['vsoma'])
    if 'vdend' in variables:
        Vdend.append(variables['vdend'])    
        
Vsoma_fig = plot3_dt(Vsoma, yaxis='V (mV)', stim_time = 150, sim_time=sim_time, title='somatic voltage', lwd=lwd,
                     yrange=[-90, -59], yabline = [-86, -60], black_trace=0, gray_trace=1, black_shift=100,
                     x_err_bar=50, y_err_bar=5, baseline=50, dt=dt, ds=ds_plot, height=height2, width=width2)        

Vdend_fig = plot3_dt(Vdend, yaxis='V (mV)', stim_time = 150, sim_time=sim_time, title='dendritic voltage', lwd=lwd,
                     yrange=[-86, -20], yabline = [-86, -30], black_trace=0, gray_trace=1, black_shift=100,
                     x_err_bar=50, y_err_bar=10, baseline=50, dt=dt, ds=ds_plot, height=height2, width=width2)

Vsoma_fig.show(config={"toImageButtonOptions": {"format": "svg"}})
Vdend_fig.show(config={"toImageButtonOptions": {"format": "svg"}})

if save:
    Vsoma_fig.write_image(f"{sim_image_path}/Vsoma.svg", format='svg', scale=3)
    Vdend_fig.write_image(f"{sim_image_path}/Vdend.svg", format='svg', scale=3)


In [ ]:
if downsample:
    # reconstruct 
    timing_range = range(120,301,10) 
    if sim in ['2251', '2253', '2255', '2257', '2259']:
        timing_range = range(120,301,1)

    timing_range = np.insert(timing_range, 0, [150, 150])

else:
    timing_range = next(iter(sim_data.values()))['metadata']['timing_range']


stimulation_paradigm_fig = plot3_timing(timing_range=timing_range, sim_time=sim_time, black_trace=0, gray_trace=1,
                 palette='oleron', alpha=1, reverse=False, black_shift=100, baseline=0,
                 width=width1, height=200, lwd=lwd, tick_height=5, title='Stimulus timing (ms)')

stimulation_paradigm_fig.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    stimulation_paradigm_fig.write_image(f"{sim_image_path}/stimulation_paradigm.svg", format='svg', scale=3)

In [ ]:
record_dendrite = next(iter(sim_data.values()))['metadata']['record_dendrite']
record_location = next(iter(sim_data.values()))['metadata']['record_location']

# get indices for the recording location
record_coords = get_coord_index_interp(cell_coordinates=cell_coordinates, target_dendrite=record_dendrite, target_location=record_location)
idx = get_coord_index(cell_coordinates=cell_coordinates, target_dendrite=record_dendrite, target_location=record_location)

compare_synapse_coords("Glutamatergic", record_dendrite, record_location, record_coords)

compare_synapse_coords("Glutamatergic", record_dendrite, record_location, cell_coordinates[idx][0:5,])


#### Note
The values obtained by interpolation are exact as the model places spines at an exact equally spaced locations. 
However **NEURON** breaks each section of the cell into sections. As there are more glutamatergic spines than segments, some spines will be binned into the nearest segment. All inputs within that bin are now functionally identical and very small differences due to the minor differences in location are ignored. In other words, the spine at the cell coordinates given by the index *idx* is functionally identical to the interpolated one. Interpolation is simply provided for plotting purposes.

In [ ]:
# Simple output traces can also be obtained from 'v_all_3D'
idx1 = get_coord_index(cell_coordinates=cell_coordinates, target_dendrite='soma[0]', target_location=0.5)
idx2 = get_coord_index(cell_coordinates=cell_coordinates, target_dendrite=record_dendrite, target_location=record_location)

Vsoma1 = []
Vdend1 = []

for sim_vars in sim_data.values():
    v_all = sim_vars['v_all_3D']
    for sim_block in v_all.values():           
        v_arrays = sim_block['v']              
        Vsoma1.append(v_arrays[idx1])
        Vdend1.append(v_arrays[idx2])
        
Vsoma1_fig = plot3_dt(Vsoma1, yaxis='V (mV)', stim_time = 150, sim_time=sim_time, title='somatic voltage', lwd=lwd,
                     yrange=[-90, -55], yabline = [-86, -60], black_trace=0, gray_trace=1, black_shift=100,
                     x_err_bar=50, y_err_bar=5, baseline=50, dt=dt, ds=ds_plot, height=height2, width=width2)        

Vdend1_fig = plot3_dt(Vdend1, yaxis='V (mV)', stim_time = 150, sim_time=sim_time, title='dendritic voltage', lwd=lwd,
                     yrange=[-86, -20], yabline = [-86, -30], black_trace=0, gray_trace=1, black_shift=100,
                     x_err_bar=50, y_err_bar=10, baseline=50, dt=dt, ds=ds_plot, height=height2, width=width2)   

Vsoma1_fig.show(config={"toImageButtonOptions": {"format": "svg"}})
Vdend1_fig.show(config={"toImageButtonOptions": {"format": "svg"}})


In [ ]:
# 3D heat map
idx = 0

Vpeaks_3D  = []

for sim_vars in sim_data.values():
    v_all = sim_vars['v_all_3D']
    max_vals = []
    for sim_block in v_all.values():
        v_arrays = sim_block['v']                 # shape: (n_traces, n_timepoints)
        # get maximum of each trace
        max_vals.extend([np.max(trace) for trace in v_arrays])
    Vpeaks_3D.append(np.array(max_vals))

# be careful with too much downsampling; it can really alter the image
V2D_fig  = heatmap2D(cell_coordinates=cell_coordinates, dend_tree=dend_tree, z=Vpeaks_3D[idx], palette='oleron', reverse=False, alpha=1, 
                     lwd=lwd, show_bar=True, title='', zmin=-85, zmax=-30, height=height1, width=width1, scale_bar=50, 
                     x_range=[-125, 175], y_range=[-150, 150], plane=plane, mirror=False, s=s, ds=2)

V2D_path_fig  = heatmap2D(cell_coordinates=cell_coordinates, dend_tree=dend_tree, dend_name=dend_name, z=Vpeaks_3D[idx], palette='oleron', reverse=False, alpha=1, 
                     lwd=lwd, show_bar=True, title='', zmin=-85, zmax=-30, height=height1, width=width1, scale_bar=50, 
                     x_range=[-125, 175], y_range=[-150, 150], plane=plane, mirror=False, s=s, ds=2)


V2D_fig.show(config={"toImageButtonOptions": {"format": "svg"}})
V2D_path_fig.show(config={"toImageButtonOptions": {"format": "svg"}})

if save:
    V2D_fig.write_image(f"{sim_image_path}/V2D_{idx}.svg", format='svg', scale=3)
    V2D_path_fig.write_image(f"{sim_image_path}/V2D_path_{idx}.svg", format='svg', scale=3)


In [ ]:
root_name = dend_name
target_dend = 'dend[15]'
extracted = extract_dend_to_target(dend_tree, root_name, target_dend)
extracted

# Rebuild idxs from the new extracted structure
idxs = []
for outer_list in extracted:
    outer_idxs = []
    for path in outer_list:
        path_idxs = []
        for dend in path:
            dendname = dend.name()
            idx = get_coord_index(cell_coordinates, dendname, None)
            path_idxs.extend(idx)
        outer_idxs.append(path_idxs)
    idxs.append(outer_idxs)

# Rebuild dists
dists = []
for outer_list in idxs:
    outer_dists = []
    for path_idxs in outer_list:
        path_dists = cell_coordinates[path_idxs, 5].astype(float)
        outer_dists.append(path_dists)
    dists.append(outer_dists)


# Build Vpeaks (use 0 for first simulation, not idx)
sim_idx = 0

Vpeaks = [[Vpeaks_3D[sim_idx][np.array(path_idxs)] for path_idxs in outer_list] for outer_list in idxs]

titles = [path[-1].name() for outer_list in extracted for path in outer_list]

fig = plot_v_mpl(dists[0], Vpeaks[0], titles=titles, colors='grey', xrange=[-2, 225], yrange=[-85, -20], height=400, width=700,  points_size=4)

fig

if save:
    fig.savefig(f"{sim_image_path}/voltage_vs_distance_{sim_idx}.svg", format='svg', dpi=300, bbox_inches='tight')



# fig = plot_v(dists[0], Vpeaks[0], titles=titles, colors='grey', yrange=[-85, -20], height=400, width=700)

# fig.show(config={"toImageButtonOptions": {"format": "svg"}})

# if save:
#     fig.write_image(f"{sim_image_path}/voltage_vs_distance_{sim_idx}.svg", format='svg', scale=3)

    

In [ ]:
delta_t = +10

tgaba = next(iter(sim_data.values()))['metadata']['stim_time']

idx = np.argmin(np.abs(np.array(timing_range) - (tgaba + delta_t)))

V2D_fig  = heatmap2D(cell_coordinates=cell_coordinates, dend_tree=dend_tree, z=Vpeaks_3D[idx], palette='oleron', reverse=False, alpha=1, 
                     lwd=lwd, show_bar=True, title='', zmin=-85, zmax=-30, height=height1, width=width1, scale_bar=50, 
                     x_range=[-125, 175], y_range=[-150, 150], plane=plane, mirror=mirror, s=s, ds=2)

V2D_path_fig  = heatmap2D(cell_coordinates=cell_coordinates, dend_tree=dend_tree, dend_name=dend_name, z=Vpeaks_3D[idx], palette='oleron', reverse=False, alpha=1, 
                     lwd=lwd, show_bar=True, title='', zmin=-85, zmax=-30, height=height1, width=width1, scale_bar=50, 
                     x_range=[-125, 175], y_range=[-150, 150], plane=plane, mirror=False, s=s, ds=2)


V2D_fig.show(config={"toImageButtonOptions": {"format": "svg"}})
V2D_path_fig.show(config={"toImageButtonOptions": {"format": "svg"}})

if save:
    V2D_fig.write_image(f"{sim_image_path}/V2D_{idx}.svg", format='svg', scale=3)
    V2D_path_fig.write_image(f"{sim_image_path}/V2D_path_{idx}.svg", format='svg', scale=3)

In [ ]:
# Build Vpeaks (use 0 for first simulation, not idx)
sim_idx = np.argmin(np.abs(np.array(timing_range) - (tgaba + delta_t)))

Vpeaks = [[Vpeaks_3D[sim_idx][np.array(path_idxs)] for path_idxs in outer_list] for outer_list in idxs]

titles = [path[-1].name() for outer_list in extracted for path in outer_list]


fig = plot_v_mpl(dists[0], Vpeaks[0], titles=titles, colors='grey', xrange=[-2, 225], yrange=[-85, -20], height=400, width=700,  points_size=4)

fig

if save:
    fig.savefig(f"{sim_image_path}/voltage_vs_distance_{sim_idx}.svg", format='svg', dpi=300, bbox_inches='tight')


In [ ]:
# units mA/cm²
summarise_sim_entry(sim_data, 'i_mechs_3D')
    

<h3 style="text-align: center;">Mechanism Current Extraction</h3>

<span style="font-family: monospace; font-weight: 550;">
get_mechs2display(sim_data, entry_name='i_mechs_3D')
</span><br>
Returns the list of ionic mechanisms recorded in the simulation data under the given entry name.<br>
Inputs: <i>sim_data (nested dictionary of simulation outputs), entry_name (string; key containing mechanism data)</i><br>
Returns: <i>List of mechanism names (e.g. ['kaf', 'kas', 'kir', ...])</i><br><br>

<span style="font-family: monospace; font-weight: 550;">
extract_mech_currents(sim_data, mech_name, idxs, entry_name='i_mechs_3D')
</span><br>
Extracts ionic currents for a specified mechanism (e.g. 'kir') across one or more coordinates (e.g. soma and dendrite) for all simulations.<br>
Inputs: <i>sim_data (nested dictionary of simulation outputs), mech_name (string), idxs (list of coordinate indices), entry_name (string; key containing mechanism data)</i><br>
Returns: <i>Dictionary mapping each index to a list of current traces for the chosen mechanism</i><br>
All extracted currents are recorded as <span style="font-weight: 600;">current densities in mA/cm^2</span>, <span style="font-weight: 600;">NEURON</span>'s units for ionic mechanisms.<br><br>

<span style="font-family: monospace; font-weight: 550;">
get_y_range(traces, dp=1)
</span><br>
Calculates a sensible y-axis range from multiple trace arrays. Rounds ymin and ymax to <i>dp</i> decimal places.<br>
Inputs: <i>traces (list or iterable of numeric arrays), dp (int; number of decimal places to round)</i><br>
Returns: <i>List [ymin, ymax] giving the rounded axis limits</i>

```python
mechs2display = get_mechs2display(sim_data)
idx1 = get_coord_index(cell_coordinates, 'soma[0]', 0.5)
idx2 = get_coord_index(cell_coordinates, record_dendrite, record_location)
I_all = extract_mech_currents(sim_data, 'kir', [idx1, idx2])
I_soma_kir = I_all[idx1]
I_dend_kir = I_all[idx2]

# convert to uA/cm^2
I_soma_kir = [trace * 1e3 for trace in I_soma_kir]

yrange = get_y_range(I_soma_kir, dp=1)

I_soma_kir_fig = plot3_dt(I_soma_kir, yaxis='', yrange=yrange, stim_time=150, sim_time=sim_time, title='', black_trace=None, y_err_bar=0.2, x_err_bar=50, baseline=50, dt=dt, ds=10, offset=True, offset_ms=150, offset_y=None)

In [ ]:
mechs2display = get_mechs2display(sim_data)
name_map = {
    'kaf': 'K<sub>v</sub>4',
    'kas': 'K<sub>v</sub>1',
    'kir': 'K<sub>ir</sub>',
    'kcnq': 'K<sub>v</sub>7'
}


idx1 = get_coord_index(cell_coordinates, 'soma[0]', 0.5)
idx2 = get_coord_index(cell_coordinates, record_dendrite, record_location)

for mech_name in mechs2display:
    I_all = extract_mech_currents(sim_data, mech_name, [idx1, idx2])
    soma_traces = I_all[idx1]
    dend_traces = I_all[idx2]

    # convert to µA/cm²
    soma_traces = [trace * 1e3 for trace in soma_traces]
    dend_traces = [trace * 1e3 for trace in dend_traces]

    # compute yrange across both soma & dend
    yrange_soma = get_y_range(soma_traces)
    yrange_dend = get_y_range(dend_traces)

    span_soma = yrange_soma[1] - yrange_soma[0]
    y_err_bar_soma = span_soma * 0.10
    y_err_bar_soma = roundup(y_err_bar_soma)
    
    span_dend = yrange_dend[1] - yrange_dend[0]
    y_err_bar_dend = span_dend * 0.10
    y_err_bar_dend = roundup(y_err_bar_dend)

    mech_label = name_map.get(mech_name, f'K<sub>{mech_name}</sub>')
    
    soma_fig = plot3_dt(soma_traces, yaxis='', yrange=yrange_soma, stim_time=150, sim_time=sim_time, height=height2, width=width2,
                        title=f'somatic {mech_label} density (µA/cm²)', y_err_bar_units='µA/cm²',
                        black_trace=0, gray_trace=1, black_shift=100, y_err_bar=y_err_bar_soma, 
                        x_err_bar=50, baseline=50, dt=dt, ds=ds_plot)

    dend_fig = plot3_dt(dend_traces, yaxis='', yrange=yrange_dend, stim_time=150, sim_time=sim_time, height=height2, width=width2, 
                        title=f'dendritic {mech_label} density (µA/cm²)', y_err_bar_units='µA/cm²',
                        black_trace=0, gray_trace=1, black_shift=100, y_err_bar=y_err_bar_dend, 
                        x_err_bar=50, baseline=50, dt=dt, ds=ds_plot)

    soma_fig.show(config={"toImageButtonOptions": {"format": "svg"}})
    dend_fig.show(config={"toImageButtonOptions": {"format": "svg"}})

    if save:
        soma_fig.write_image(f"{sim_image_path}/current_density_soma_{mech_name}.svg", format='svg', scale=3)
        dend_fig.write_image(f"{sim_image_path}/current_density_dend_{mech_name}.svg", format='svg', scale=3)